In [4]:
import torch
import cv2
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from transformers import pipeline

In [8]:
def load_and_normalize_images(image_paths):
    """
    Load and normalize all images to the same size while maintaining aspect ratios.
    We first determine the target size based on the smallest image dimension,
    then resize all images accordingly with proper padding.
    """
    print("\nLoading and normalizing images...")
    
    # First, load all images and get their dimensions
    images = {}
    dimensions = []
    for view_type, path in image_paths.items():
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to load image: {path}")
        images[view_type] = img
        dimensions.append(img.shape[:2])
        print(f"{view_type.capitalize()} view original size: {img.shape[1]}x{img.shape[0]}")
    
    # Find the smallest dimension across all images
    min_dim = min(min(dim) for dim in dimensions)
    # Round to nearest multiple of 32 (common requirement for deep learning models)
    target_size = ((min_dim // 32) * 32)
    print(f"\nTarget size determined: {target_size}x{target_size}")
    
    normalized_images = {}
    for view_type, img in images.items():
        # Calculate aspect ratio preserving dimensions
        h, w = img.shape[:2]
        aspect = w / h
        
        if aspect > 1:
            new_w = target_size
            new_h = int(target_size / aspect)
        else:
            new_h = target_size
            new_w = int(target_size * aspect)
            
        # Resize image
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        # Create square canvas with padding
        square_img = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        
        # Calculate padding to center the image
        pad_y = (target_size - new_h) // 2
        pad_x = (target_size - new_w) // 2
        
        # Place resized image in center
        square_img[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = resized
        
        normalized_images[view_type] = square_img
        print(f"{view_type.capitalize()} view normalized size: {square_img.shape[1]}x{square_img.shape[0]}")
    
    return normalized_images, target_size

In [12]:
def create_object_mesh(image_input, depth_threshold=0.75):
    """
    Creates a 3D mesh from either an image path or a numpy array.
    
    Args:
        image_input: Can be either a file path (str) or a pre-loaded image (numpy array)
        depth_threshold: Threshold for depth filtering (0.0 to 1.0)
    """
    # Handle input flexibly - either load from file or use provided array
    if isinstance(image_input, str):
        image = cv2.imread(image_input)
        if image is None:
            raise ValueError(f"Failed to load image from path: {image_input}")
    else:
        image = image_input  # Already a numpy array
        
    # Continue with edge detection and processing as before
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(image_gray, 100, 200)
    kernel = np.ones((5, 5), np.uint8)
    dilated_edges = cv2.dilate(edges, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
        
    largest_contour = max(contours, key=cv2.contourArea)
    mask = np.zeros(image_gray.shape, dtype=np.uint8)
    cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
    mask_blurred = cv2.GaussianBlur(mask, (1, 1), 0)
    
    output_image = np.zeros_like(image)
    output_image[mask_blurred > 0] = image[mask_blurred > 0]
    
    # Get depth map using Depth Anything model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    depth_model = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-base-hf", device=device)
    
    # Convert to PIL Image for depth estimation
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image_rgb)
    depth_predictions = depth_model(image_pil)
    depth_map = np.array(depth_predictions["depth"])
    
    # Process depth map
    height, width, _ = output_image.shape
    depth_map_resized = cv2.resize(depth_map, (width, height))
    
    # Apply mask to depth map
    mask_resized = cv2.resize(mask, (depth_map_resized.shape[1], depth_map_resized.shape[0]))
    mask_contour = mask_resized > 0
    depth_edges = np.zeros_like(depth_map_resized)
    depth_edges[mask_contour] = depth_map_resized[mask_contour]
    depth_edges_smoothed = cv2.GaussianBlur(depth_edges, (5, 5), 0)
    
    # Normalize depth values
    depth_norm = (depth_edges_smoothed - depth_edges_smoothed.min()) / (depth_edges_smoothed.max() - depth_edges_smoothed.min())
    
    # Convert to RGB for visualization
    output_rgb = cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB)
    
    # Create coordinate grids
    x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))
    
    # Flatten arrays
    x_coords_flat = x_coords.ravel().tolist()
    y_coords_flat = y_coords.ravel().tolist()
    z_coords_flat = depth_norm.ravel().tolist()
    colors_flat = output_rgb.reshape(-1, 3).tolist()
    
    # Apply depth threshold and create mask for valid points
    mask = []
    for z, color in zip(z_coords_flat, colors_flat):
        is_valid = (z > depth_threshold) and not (color[0] == 0 and color[1] == 0 and color[2] == 0)
        mask.append(is_valid)
    
    # Filter points based on mask
    x_filtered = [x for x, m in zip(x_coords_flat, mask) if m]
    y_filtered = [y for y, m in zip(y_coords_flat, mask) if m]
    z_filtered = [z for z, m in zip(z_coords_flat, mask) if m]
    colors_filtered = [c for c, m in zip(colors_flat, mask) if m]
    
    # Create the Plotly figure
    fig = go.Figure(data=[go.Scatter3d(
        x=x_filtered,
        y=y_filtered,
        z=z_filtered,
        mode='markers',
        marker=dict(
            size=2,
            color=[f'rgb({r}, {g}, {b})' for r, g, b in colors_filtered],
            opacity=1
        )
    )])
    
    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(nticks=10, range=[0, width]),
            yaxis=dict(nticks=10, range=[0, height]),
            zaxis=dict(nticks=10, range=[0, max(z_filtered)]),
        ),
        margin=dict(l=0, r=0, b=0, t=30)
    )
    
    return fig

In [16]:

# Process all three views
folder = "../static/uploads/test-ring"
image_paths = {
    'front': f"{folder}/front.png",
    'back': f"{folder}/back.png",
    'top': f"{folder}/top.png"
}


# First normalize all images
try:
    normalized_images, target_size = load_and_normalize_images(image_paths)
    print("\nAll images successfully normalized to size:", target_size)

    # First, let's get our three normalized meshes individually
    front_fig = create_object_mesh(normalized_images['front'])

    if front_fig is not None:
        front_fig.update_layout(
            title=dict(
                text=f"3D Mesh - front View (Size: {target_size}x{target_size})",
                y=0.95
            )
        )
        front_fig.show()
    else:
        print(f"Failed to create mesh for front view")

    back_fig = create_object_mesh(normalized_images['back'])
    if back_fig is not None:
        back_fig.update_layout(
            title=dict(
                text=f"3D Mesh - back View (Size: {target_size}x{target_size})",
                y=0.95
            )
        )
        back_fig.show()
    else:
        print(f"Failed to create mesh for back view")

    
    top_fig = create_object_mesh(normalized_images['top'])
    if top_fig is not None:
        top_fig.update_layout(
            title=dict(
                text=f"3D Mesh - top View (Size: {target_size}x{target_size})",
                y=0.95
            )
        )
        top_fig.show()
    else:
        print(f"Failed to create mesh for top view")
            
except Exception as e:
    print(f"Error during image normalization: {str(e)}")


Loading and normalizing images...
Front view original size: 225x196
Back view original size: 214x189
Top view original size: 881x876

Target size determined: 160x160
Front view normalized size: 160x160
Back view normalized size: 160x160
Top view normalized size: 160x160

All images successfully normalized to size: 160


In [15]:
from sklearn.neighbors import KDTree

In [29]:
# Let's add some diagnostic printing to understand our data structure
def extract_points_from_fig(fig):
    """
    Extracts point coordinates and colors from a Plotly figure with detailed validation.
    """
    data = fig.data[0]
    
    # Print the data structure for debugging
    print("\nExtracting data from figure:")
    print(f"X points shape: {np.array(data.x).shape}")
    print(f"Y points shape: {np.array(data.y).shape}")
    print(f"Z points shape: {np.array(data.z).shape}")
    print(f"Sample of first few points:")
    for i in range(min(5, len(data.x))):
        print(f"Point {i}: ({data.x[i]}, {data.y[i]}, {data.z[i]})")
    
    points = {
        'x': np.array(data.x),
        'y': np.array(data.y),
        'z': np.array(data.z),
        'colors': list(data.marker.color)
    }
    return points

def combine_orthogonal_meshes(front_points, back_points, top_points, consistency_threshold=1):
    print("\nStarting mesh combination process...")
    
    # First create our coordinate arrays
    front_coords = np.column_stack([front_points['x'], front_points['y'], front_points['z']])
    back_coords = np.column_stack([back_points['x'], back_points['y'], back_points['z']])
    top_coords = np.column_stack([top_points['x'], top_points['y'], top_points['z']])

    
    print(f"Coordinate arrays created:")
    print(f"Front shape: {front_coords.shape}")
    print(f"Back shape: {back_coords.shape}")
    print(f"Top shape: {top_coords.shape}")
    
    # Create KD-trees for spatial matching
    front_tree = KDTree(front_coords[:, :2])  # XY plane
    back_tree = KDTree(back_coords[:, :2])    # XY plane
    top_tree = KDTree(np.column_stack([top_coords[:, 0], top_coords[:, 2]]))  # XZ plane
    
    consistent_points = []
    point_colors = []

    # Normalize X and Y coordinates to range [0, target_size]
    def normalize_spatial_coords(coords, target_size):
        """
        Normalize coordinates while maintaining aspect ratio and scale.
        X and Y should be in range [0, target_size]
        Z should remain in range [0, 1]
        """
        result = coords.copy()
        # Normalize X and Y to target size
        result[:, 0] = (coords[:, 0] / coords[:, 0].max()) * target_size  # X coordinates
        result[:, 1] = (coords[:, 1] / coords[:, 1].max()) * target_size  # Y coordinates
        # Z coordinates are already normalized 0-1, keep them that way
        return result
    
    # Normalize all coordinate sets
    front_coords = normalize_spatial_coords(front_coords, target_size)
    back_coords = normalize_spatial_coords(back_coords, target_size)
    top_coords = normalize_spatial_coords(top_coords, target_size)
    
    print("Coordinate ranges after normalization:")
    print(f"Front X: [{front_coords[:, 0].min():.1f}, {front_coords[:, 0].max():.1f}]")
    print(f"Front Y: [{front_coords[:, 1].min():.1f}, {front_coords[:, 1].max():.1f}]")
    print(f"Front Z: [{front_coords[:, 2].min():.3f}, {front_coords[:, 2].max():.3f}]")
    
    
    # Process each point from the front view
    for i in range(len(front_coords)):
        if i % 100 == 0:
            print(f"Processing point {i}/{len(front_coords)}")
        
        # Get the current point we're processing
        front_point = front_coords[i]
        
        # Prepare queries for matching
        xy_query = front_point[:2].reshape(1, -1)  # For back view matching
        xz_query = np.array([[front_point[0], front_point[2]]])  # For top view matching
        
        # Find nearest neighbors in back and top views
        back_dists, back_indices = back_tree.query(xy_query, k=1)
        top_dists, top_indices = top_tree.query(xz_query, k=1)
        
        # Get the actual indices from the query results
        back_idx = back_indices[0][0]  # Extract the single index
        top_idx = top_indices[0][0]    # Extract the single index
        
        # Check if matches are within our threshold
        if back_dists[0][0] > consistency_threshold or top_dists[0][0] > consistency_threshold:
            continue
            
        # Get the matched points using correct indexing
        back_point = back_coords[back_idx]
        top_point = top_coords[top_idx]
        
        # Create consensus point with normalized coordinates
        consensus_point = np.array([
            front_point[0],                    # X from front view (already normalized)
            (front_point[1] + top_point[1])/2, # Average Y (already normalized)
            front_point[2]                     # Keep Z in 0-1 range
        ])
        
        # Store the point and its color
        consistent_points.append(consensus_point)
        point_colors.append(front_points['colors'][i])
    
    if not consistent_points:
        print("No consistent points found. Try adjusting the threshold.")
        return None, None
    
    consistent_points = np.array(consistent_points)
    print(f"\nFound {len(consistent_points)} consistent points")
    
    # Create the visualization
    fig = go.Figure(data=[go.Scatter3d(
        x=consistent_points[:, 0],
        y=consistent_points[:, 1],
        z=consistent_points[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=point_colors,
            opacity=1
        )
    )])
    
    fig.update_layout(
        title='Combined 3D Mesh from Orthogonal Views',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        margin=dict(l=0, r=0, b=0, t=30)
    )
    
    return fig, consistent_points

In [33]:

# Now we can use our existing meshes
front_points = extract_points_from_fig(front_fig)
back_points = extract_points_from_fig(back_fig)
top_points = extract_points_from_fig(top_fig)

# Combine them while preserving color information
combined_fig, combined_points = combine_orthogonal_meshes(
    front_points, 
    back_points, 
    top_points,
    consistency_threshold=10000000000000
)

# Display the result
combined_fig.show()


Extracting data from figure:
X points shape: (1042,)
Y points shape: (1042,)
Z points shape: (1042,)
Sample of first few points:
Point 0: (84, 55, 0.7525773195876289)
Point 1: (82, 56, 0.7525773195876289)
Point 2: (83, 56, 0.7835051546391752)
Point 3: (84, 56, 0.7783505154639175)
Point 4: (81, 57, 0.7835051546391752)

Extracting data from figure:
X points shape: (2399,)
Y points shape: (2399,)
Z points shape: (2399,)
Sample of first few points:
Point 0: (81, 71, 0.788659793814433)
Point 1: (82, 71, 0.7938144329896907)
Point 2: (83, 71, 0.7525773195876289)
Point 3: (79, 72, 0.788659793814433)
Point 4: (80, 72, 0.8762886597938144)

Extracting data from figure:
X points shape: (1260,)
Y points shape: (1260,)
Z points shape: (1260,)
Sample of first few points:
Point 0: (73, 32, 0.7580645161290323)
Point 1: (74, 32, 0.7661290322580645)
Point 2: (75, 32, 0.7701612903225806)
Point 3: (76, 32, 0.7701612903225806)
Point 4: (77, 32, 0.7701612903225806)

Starting mesh combination process...
Coor